# AmSC IRI Job Submission (Networking Optional)

AmSC Resource Orchestration Toolkit (AmSCROT) - Orchestrating Infrastructure Service capabilities.

This notebook demonstrates how to submit compute jobs to **ESnet IRI East** and **ESnet IRI West** sites using the `amscrot` Client module, with optional SENSE network provisioning between the sites. 

## Workflow Overview

1. **Initialize** the AmSCROT client
2. **Create** a session
3. *(Optional)* **Provision** a SENSE L2VPN network between sites
4. **Set up** ESnet IRI service clients for East and West
5. **Discover** available compute resources at each site
6. **Define** and submit batch jobs
7. **Monitor** job status
8. **Clean up**

## Prerequisites

### Required Packages
- `amscrot-py==1.0.0` (installed)
- `globus-sdk` (installed)
- `dotenv` (installed)

### Installation

```
pip install amscrot-py globus-sdk dotenv
```

### Credentials

ESnet IRI service clients authenticate via API keys stored in `~/.amscrot/credentials.yml`. You need entries for **both** the East and West sites:

```yaml
# ~/.amscrot/credentials.yml

esnet-iri-east:
  api_key: <YOUR_BEARER_TOKEN>
  api_endpoint: https://iri-dev.ppg.es.net

esnet-iri-west:
  api_key: <YOUR_BEARER_TOKEN>
  api_endpoint: https://esnet-west.sdn-sense.net/
```

If you also want SENSE network provisioning, add a `sense` section, for example::

```yaml
sense:
  AUTH_ENDPOINT: https://sense-o.es.net:8543/auth/realms/StackV/protocol/openid-connect/token
  API_ENDPOINT: https://sense-o-dev.es.net:8443/StackV-web/restapi
  CLIENT_ID: Portal
  USERNAME: <your-username>
  PASSWORD: <your-password>
  SECRET: None
  verify: False
```

---
## 0. Use Globus Auth Token

In [ ]:
import globus_sdk
import datetime
import time
import os
import yaml
from dotenv import load_dotenv


load_dotenv()  # take environment variables from .env file
GLOBUS_ID = os.getenv("GLOBUS_ID")
GLOBUS_SECRET = os.getenv("GLOBUS_SECRET")

print ("Using Globus ID", GLOBUS_ID)

# Create a confidential client
client = globus_sdk.ConfidentialAppAuthClient(GLOBUS_ID, GLOBUS_SECRET)
# Start the OAuth flow
client.oauth2_start_flow(
    redirect_uri="http://localhost:5000/callback",  # or your registered redirect URI
    requested_scopes=["openid", "profile", "email", "urn:globus:auth:scope:auth.globus.org:view_identities"]
)
# Get the authorization URL
authorize_url = client.oauth2_get_authorize_url()
print(f"Visit this URL in your browser:\n{authorize_url}\n")

# After visiting the URL and authorizing, you'll be redirected to a URL with a code parameter
auth_code = input("Paste the 'code' parameter from the redirect URL: ")

# Exchange the code for tokens

token_response = client.oauth2_exchange_code_for_tokens(auth_code)
access_token_data = token_response.by_resource_server['auth.globus.org']
access_token = access_token_data['access_token']
expires_at = access_token_data['expires_at_seconds']

# Convert to human-readable time
expiration_time = datetime.datetime.fromtimestamp(expires_at)
print(f"Token expires at: {expiration_time}")

# Calculate how long until expiration
seconds_until_expiration = expires_at - time.time()
hours_until_expiration = seconds_until_expiration / 3600

print(f"Token expires in {hours_until_expiration:.2f} hours")
# print("---------------------")
# print(access_token_data)
# print("---------------------")
print(f"\nYour access token:\n{access_token}")

# Create credentials data
credentials = {
    'esnet-iri-east': {
        'api_key': access_token,
        'api_endpoint': 'https://iri-dev.ppg.es.net'
    },
    'esnet-iri-west': {
        'api_key': access_token,
        'api_endpoint': 'https://esnet-west.sdn-sense.net/'
    }
}

# Write to ~/.amscrot/credentials-new.yml
cred_path = os.path.expanduser("~/.amscrot/credentials-new.yml")
os.makedirs(os.path.dirname(cred_path), exist_ok=True)
with open(cred_path, 'w') as f:
    yaml.dump(credentials, f)
print(f"Wrote credentials to {cred_path}")

---
## 0.1 Use SENSE Auth Token (Optional) 

If you already have a SENSE-O credentials file, you can use it to generate a token that works with the ESnet IRI endpoints.

In [ ]:
import os
import yaml
from sense.client.apiclient import ApiClient
from sense.common import getConfig

# Force use of the new auth file
os.environ['SENSE_AUTH_OVERRIDE'] = os.path.expanduser("~/.sense-o-auth-new.yaml")

# Get config and token
config = getConfig()
sense_client = ApiClient(config)
token = sense_client.token['access_token']
print(f"Got SENSE token: {token[:10]}...")

# Create credentials data
credentials = {
    'sense': {
        'AUTH_ENDPOINT': config.get('AUTH_ENDPOINT'),
        'API_ENDPOINT': config.get('API_ENDPOINT'),
        'CLIENT_ID': config.get('CLIENT_ID'),
        'USERNAME': config.get('USERNAME'),
        'PASSWORD': config.get('PASSWORD'),
        'SECRET': config.get('SECRET'),
        'verify': config.get('verify', False)
    },
    'esnet-iri-east': {
        'api_key': token,
        'api_endpoint': 'https://iri-dev.ppg.es.net'
    },
    'esnet-iri-west': {
        'api_key': token,
        'api_endpoint': 'https://esnet-west.sdn-sense.net/'
    }
}

# Write to ~/.amscrot/credentials-new.yml
cred_path = os.path.expanduser("~/.amscrot/credentials-new.yml")
os.makedirs(os.path.dirname(cred_path), exist_ok=True)
with open(cred_path, 'w') as f:
    yaml.dump(credentials, f)
print(f"Wrote credentials to {cred_path}")

---
## 1. Initialize Client & Session

In [ ]:
import time
from amscrot.client.client import Client
from amscrot.client.job import Job, JobType, JobServiceType, JobSpec
from amscrot.serviceclient import ServiceClient
from amscrot.util.constants import Constants

client = Client()
session = client.create_session("sense-networked-jobs")
print("Client and session initialized.")

## 2. (Optional) Add SENSE Network

Set `USE_NETWORK = True` to provision a SENSE L2VPN between the East and West sites. This requires valid SENSE credentials in your credentials file.

In [ ]:
USE_NETWORK = False  # Set to True to enable SENSE network provisioning

if USE_NETWORK:
    sense_provider = client.add_provider(
        label="sense",
        type="sense",
        name="sense-provider",
        profile="sense",
        credential_file="~/.amscrot/credentials-new.yml"
    )
    net1 = session.add_network(
        label="net1",
        provider=sense_provider,
        name_prefix="test-net",
        site="ESnet",
        profile="AmSC-WFC-L2VPN",
        count=1
    )
    print("SENSE network added to session.")
else:
    print("Skipping SENSE network provisioning.")

## 3. Set Up ESnet IRI Service Clients

We create two service clients — one for each IRI site. Each client loads its credentials from the corresponding profile in `~/.amscrot/credentials-new.yml`.

In [ ]:
east_client = ServiceClient.create(
    type=Constants.ServiceType.ESNET_IRI,
    name="iri-east",
    profile="esnet-iri-east",
    credential_file="~/.amscrot/credentials-new.yml"
)
session.add_service_client(east_client)

west_client = ServiceClient.create(
    type=Constants.ServiceType.ESNET_IRI,
    name="iri-west",
    profile="esnet-iri-west",
    credential_file="~/.amscrot/credentials-new.yml"
)
session.add_service_client(west_client)

## 4. Discover Compute Resources

Each service client's `discover()` method returns a `DiscoveryResult` container with typed accessors for each resource type. We use `.compute` to find available compute resources at each site.

In [ ]:
east_discovery = east_client.discover()
west_discovery = west_client.discover()

print(f"East discovery: {east_discovery.summary()}")
print(f"West discovery: {west_discovery.summary()}")

assert east_discovery.compute, "No compute resources found on East site!"
assert west_discovery.compute, "No compute resources found on West site!"

east_resource_id = east_discovery.compute[0].data.get("id")
west_resource_id = west_discovery.compute[0].data.get("id")

print(f"\nEast resource_id: {east_resource_id}")
print(f"West resource_id: {west_resource_id}")

## 5. Define Job Specs & Jobs

Each job is a simple batch job that runs `/bin/echo` on the discovered compute resource. The `resource_id` is set dynamically from the discovery step above.

In [ ]:
common_resources = {
    "node_count": 1,
    "process_count": 1,
    "processes_per_node": 1,
    "cpu_cores_per_process": 1,
    "gpu_cores_per_process": 1,
    "exclusive_node_use": True,
    "memory": 268435456
}

common_attributes = {
    "directory": "/tmp",
    "duration": 60,
    "queue_name": "debug",
    "account": "interactive"
}

spec_east = JobSpec(
    executable=["/bin/echo", "Hello AmSC East"],
    resources=common_resources,
    attributes={"resource_id": east_resource_id, **common_attributes}
)

spec_west = JobSpec(
    executable=["/bin/echo", "Hello AmSC West"],
    resources=common_resources,
    attributes={"resource_id": west_resource_id, **common_attributes}
)

job1 = Job(name="job-1", type=JobType.COMPUTE, service_type=JobServiceType.BATCH,
           service_client=east_client, job_spec=spec_east)

job2 = Job(name="job-2", type=JobType.COMPUTE, service_type=JobServiceType.BATCH,
           service_client=west_client, job_spec=spec_west)

session.add_job(job1)
session.add_job(job2)

print("Jobs defined and added to session.")

## 6. Plan

The plan phase validates all resources and job specs before anything is created.

In [ ]:
session.plan()

## 7. Apply and monitor

Apply creates resources (if networking is enabled) and submits the compute jobs. 
Poll both jobs until they complete (or time out after 60 seconds).

In [ ]:
rc = session.apply()
if rc:
    raise RuntimeError(f"Session apply failed with rc={rc}")
print("Session applied successfully.")

for i in range(30):
    s1 = east_client.status(job_name="job-1")
    s2 = west_client.status(job_name="job-2")
    
    status1 = s1.get('status')
    status2 = s2.get('status')
    print(f"Poll {i}: Job1={status1}  Job2={status2}")
    
    if (status1 in ["DONE", "ERROR", "DESTROYED"] and 
        status2 in ["DONE", "ERROR", "DESTROYED"]):
        break
    
    time.sleep(2)

assert s1.get('status') == 'DONE', f"Job1 failed or timed out: {s1}"
assert s2.get('status') == 'DONE', f"Job2 failed or timed out: {s2}"
print(f"\n✅ Both jobs completed successfully!")

## 9. Clean Up

Destroy the session to tear down any provisioned resources and cancel remaining jobs.

In [ ]:
session.destroy()